# CIS 5450 Project: Difficulty Topics
**Team Members:** Jiantang Ma, Jun Yu Chen, Rita Wang, Ziyan Li

> This notebook documents how we implemented difficulty topics in our project. Use the link button in the top right when you select a cell to get a **hyperlink**.

---

## Topic 1: **Feature Engineering — Categorical Encoding and Derived Features**

### Implementation & Rationale

Our merged dataset contains many fields that cannot be used by a model directly — comma-separated genre and tag strings, ISO timestamps, list-formatted language fields, and categorical publisher class labels. Several numeric fields (e.g., Price, Achievements count, DLC count) are already model-ready in the raw data. Our engineering focused on transforming the unstructured fields into usable representations.

The most important design decision was **separating post-release features from launch-time features**. Signals like review scores, playtime, recommendation counts, and estimated owner ranges are strong predictors but constitute data leakage for a pre-launch model — they cannot be known before a game ships. We implemented a `post_release` flag in the feature pipeline to enforce this separation, ensuring all launch-time models are trained only on information available before release.

Beyond leakage prevention, our engineering covers: **ordinal encoding** of publisher class (preserving its real-world rank ordering), **binary parsing** of genres, Steam categories, and user tags into indicator columns, **count and frequency features** for language support and publisher/developer activity, **temporal decomposition** of release dates into year/month/quarter, and a **binary missingness indicator** for Metacritic score to handle 96% zero-rate without imputation bias.

**Why we used these methods:**
Without these transformations, the most informative fields in the dataset would be entirely unusable. Each encoding choice was also motivated by domain knowledge rather than applied mechanically — ordinal encoding for publisher class preserves meaningful hierarchy that one-hot encoding would discard, and the post-release separation is what makes the model valid for its intended consulting use case.

**Results and conclusions:**
The engineered features are central to model performance. `publisher_class_ord` (ordinal-encoded from a raw string label) is the strongest predictor in the OLS model (β = +2.156). Features derived from string parsing — `has_trading_cards`, `has_dlc`, `language_count` — rank among the top predictors in RFECV. `release_year` captures the market saturation trend visible in the EDA. Collectively, the engineered feature set enables the model to explain 65.2% of variance in log sales (R² = 0.652), compared to near-zero for a mean predictor.


**Why we used these methods:**
Each choice was driven by the structure of the raw field and domain knowledge. Ordinal encoding preserves the real-world hierarchy of publisher tiers. Binary parsing of genres, categories, and tags is necessary because multi-label string fields have no natural numeric representation. Frequency encoding handles high-cardinality identifiers that would be unusable as one-hot columns. The `has_metacritic` indicator is a deliberate design choice to handle near-complete missingness correctly rather than imputing or dropping.

**Results and conclusions:**
The engineered features dominate model performance. `publisher_class_ord` (ordinal-encoded from a raw string label) is the strongest predictor in the OLS model (β = +2.156) and the most stable in the bootstrap test (100% sign consistency). `has_trading_cards`, `has_dlc`, and `language_count` — all derived from raw string fields — rank among the top features in RFECV. `release_year` captures the market saturation trend visible in the EDA: without it, the model would miss a key structural pattern. Collectively, the engineered feature set enables the model to explain 65.2% of variance in log sales (R² = 0.652), compared to near-zero for a mean predictor.


---

## Topic 2: **Feature Importance — Recursive Feature Elimination with Cross-Validation (RFECV)**

### Implementation & Rationale

We implemented **Recursive Feature Elimination with Cross-Validation (RFECV)** using XGBoost as the base estimator to systematically identify the most predictive launch-time features for Steam game sales (`log(copiesSold + 1)`).

**Implementation details:**
Starting from a pool of 79 launch-time features, RFECV iteratively eliminates the least important features (by XGBoost gain-based importance) in steps of 2, evaluating every resulting subset size (79, 77, 75, …, 5 — 38 sizes total) using 3-fold cross-validation on the training set (80,633 rows). At each subset size, 3-fold CV records the mean and standard deviation of RMSE on `log(copiesSold)` across folds. Two final picks are produced:

- **`chosen_by_min` = 71 features** — the subset with the lowest mean CV RMSE (1.55219)
- **`chosen_by_1se` = 60 features** — the smallest subset within one standard error of the best (mean RMSE = 1.55485), selected via the 1-SE rule

We adopted the **1-SE pick of 60 features** for downstream modeling. The margin between 71 and 60 features (ΔRMSE = 0.00266) is smaller than the within-fold noise at n=71 (std = 0.00351), making them statistically indistinguishable while the 60-feature model is more parsimonious and interpretable.

**Why we used it:**
Our feature engineering pipeline converts raw categorical strings — genres, tags, and categories stored as comma-separated text — into binary indicator columns, one per unique value. While this encoding is necessary for modeling, it inflates the feature space significantly (from a handful of raw columns to 79 binary and numeric features). Many of these binary features are sparse and redundant, and including all of them risks adding noise without signal. RFECV provides a principled way to identify which of these engineered features actually matter: it evaluates each feature in context of the others that remain, and cross-validation ensures the selected subset generalizes rather than overfitting to a single train/test split — something simple coefficient ranking or single-model importance cannot guarantee.

**Results and conclusions:**
The CV-RMSE curve reveals three distinct regions: a steep drop from 5→15 features (each addition provides real signal), diminishing returns from 15→40, and a noise floor from 40→79 features (~1.555 ± 0.003) where adding more features produces no meaningful improvement. This confirms that the marginal predictive value of the bottom ~19 features is negligible. The feature ranking shows that `publisher_class_ord`, `release_year`, `has_trading_cards`, `Price`, and `Achievements` are among the most important predictors — consistent with the OLS regression analysis and hypothesis testing results. On the log-scale CV metric, the 71-feature model achieves the best mean RMSE (1.55219) and the 60-feature 1-SE pick is statistically tied (1.55485) while using 19 fewer features. We adopted both for downstream analysis. When applied to tree-based models, the 60-feature subset slightly outperformed the 71-feature subset in held-out test performance, further supporting the 1-SE selection as the preferred choice. On the original copies scale, both subsets outperform the full 79-feature model by decreasing RMSE by ~20,000, a meaningful improvement for a consulting use case where the goal is to estimate real commercial outcomes.

---

## Topic 3: **Hypothesis Testing — Statistical Inference and Simulation-Based Stability Validation**

### Implementation & Rationale

Our project's goal is to provide consulting-grade recommendations to game developers and investors, not just accurate predictions. Hypothesis testing lets us formally ask: *which relationships in our model are statistically real, and are they reliable enough to base business decisions on?* We implemented three hypothesis tests combining standard regression inference with simulation-based validation.

**Hypotheses 1 & 2 — Regression-based inference (OLS t-tests)**

We tested two practically motivated claims using coefficients from the OLS model (Section 3), which controls for all launch-time features:

- **H1 (Price Effect):** Does lowering price lead to more copies sold? We tested H₀: β_price = 0 against Hₐ: β_price < 0. The result was a significant *positive* coefficient — counter to the naive intuition — suggesting that within paid games, price functions as a quality signal. Developers should price confidently rather than race to the bottom.

- **H2 (Publisher Class Effect):** Can an established publisher acquiring an indie title and releasing it under their label significantly boost sales through platform advantages alone, without changing the game itself? We tested H₀: β_publisher_class = 0 against Hₐ: β_publisher_class ≠ 0. The result was highly significant, with each publisher tier associated with approximately ×8.6 more copies sold — making this the strongest single predictor in the model and providing quantitative grounding for acquisition-based strategies.

**Hypothesis 3 — Simulation-based stability test (Bootstrap, 500 iterations)**

A recommendation is only actionable if it generalizes beyond the dataset used to derive it. To validate model claims, we implemented a bootstrap resampling test: resample the full dataset with replacement 500 times, refit the OLS model each time, and examine whether key coefficients maintain consistent sign and magnitude across all resamples.

This is the simulation-based component of our hypothesis testing framework. The null hypothesis is that the effects are unstable (coefficients change sign or straddle zero across samples), and we reject it if Sign% ≥ 95% and the 95% bootstrap CI excludes zero.

All five predictors — Price, Publisher Class, Achievements, Tag Count, and Has DLC — achieved 100% sign consistency with CIs well clear of zero, leading us to reject H₀ for all five. The bootstrap means closely replicate the OLS point estimates, confirming that the effects identified in H1 and H2 are not sample-specific artifacts but stable, generalizable patterns in the Steam market.

**Why we used this approach:**
Standard regression inference (H1, H2) gives us direction and significance under distributional assumptions. The bootstrap (H3) provides assumption-free, simulation-based validation of the same claims — if both converge, the conclusions are robust. Together, they convert model coefficients into *defensible consulting advice*: developers are encouraged to make more in-game achievements and DLC even in the last minute before the game is published; investors can quantify acquisition upside knowing the publisher class effect holds across any plausible sample of the market.